# Perform segmentation and feature extraction using CellProfiler

## Import libraries

In [1]:
import argparse
import pathlib
import pprint

import sys

sys.path.append("../utils")
import cp_parallel

# check if in a jupyter notebook
try:
    cfg = get_ipython().config
    in_notebook = True
except NameError:
    in_notebook = False

## Set paths and variables

In [ ]:
# directory where the LoadData CSVs are located (one folder per cell line, one CSV per plate/time point)
loaddata_dir = pathlib.Path("./loaddata_csvs/cell_line_loaddata_csvs").resolve(
    strict=True
)

# directory where the CellProfiler pipelines are located (one pipeline per cell line)
pipeline_dir = pathlib.Path("./pipeline").resolve(strict=True)

if not in_notebook:
    print("Running as script")
    # set up arg parser
    parser = argparse.ArgumentParser(
        description="CellProfiler segmentation and feature extraction"
    )

    parser.add_argument(
        "--input_csv",
        type=str,
        help="Path to the LoadData CSV file (one cell line from one plate/time point) to process images",
    )

    args = parser.parse_args()
    loaddata_csv = pathlib.Path(args.input_csv).resolve(strict=True)
else:
    print("Running in a notebook")
    loaddata_csv = pathlib.Path(
        f"{loaddata_dir}/A673/BR00150698_A673_loaddata_with_illum.csv"
    ).resolve(strict=True)

# set the run type for the parallelization
run_name = "analysis"

# cell line is the name of the folder holding the LoadData CSV and plate (time point) is the prefix of the file name
cell_line = loaddata_csv.parent.name
plate = loaddata_csv.stem.split("_")[0]

# set path for the CellProfiler pipeline specific to the cell line
path_to_pipeline = pathlib.Path(pipeline_dir / f"analysis_{cell_line}.cppipe").resolve(
    strict=True
)

# set main output dir for all jobs if it doesn't exist
output_dir = pathlib.Path("./sqlite_outputs")
output_dir.mkdir(exist_ok=True)

## Create dictionary to process data

In [ ]:
# Name for the job is the plate (time point) and cell line so each job gets its own output folder and log
name = f"{plate}_{cell_line}"

# create plate info dictionary with all parts of the CellProfiler CLI command to run in parallel
plate_info_dictionary = {
    name: {
        "path_to_loaddata": loaddata_csv,
        "path_to_output": output_dir / name,
        "path_to_pipeline": path_to_pipeline,
    }
}

# view the dictionary to assess that all info is added correctly
pprint.pprint(plate_info_dictionary, indent=4)

## Perform segmentation and feature extraction (analysis)

Note: This code cell was not ran as we prefer to perform CellProfiler processing tasks via `sh` file (bash script) which is more stable.

In [ ]:
cp_parallel.run_cellprofiler_parallel(
    plate_info_dictionary=plate_info_dictionary, run_name=run_name
)